# ⚡ ZeroSess — Online Telegram Session String Generator
### Generate Pyrogram v2 & Telethon Session Strings in 10 Seconds

[![GitHub](https://img.shields.io/badge/GitHub-ZeroSess-181717?style=for-the-badge&logo=github)](https://github.com/saahiyo-cloud/ZeroSess)
[![Zero-PII](https://img.shields.io/badge/Security-Zero--PII%20In--Memory-brightgreen?style=for-the-badge)](https://github.com/saahiyo-cloud/ZeroSess)

🔒 **Privacy Guarantee:** This notebook runs directly in your private Google Colab runtime. No credentials or session strings are ever stored, logged, or transmitted anywhere outside official Telegram MTProto servers.

### Step 1: Install Dependencies
Click the **Run** button below to install `pyrogram`, `telethon`, and `tgcrypto`.

In [ ]:
!pip install -q pyrogram tgcrypto telethon nest_asyncio

### Step 2: Run Interactive Generator
Fill in your `API_ID` & `API_HASH` from [my.telegram.org](https://my.telegram.org) and follow the prompts.

In [ ]:
import nest_asyncio
import asyncio
import getpass
from pyrogram import Client
from telethon import TelegramClient
from telethon.sessions import StringSession

nest_asyncio.apply()

print("⚡ Welcome to ZeroSess Online Generator")
print("======================================")
lib = input("Select Library:\n[1] Pyrogram v2 (Default & Recommended)\n[2] Telethon\nChoice (1/2): ").strip() or "1"
api_id = int(input("Enter API_ID (from my.telegram.org): ").strip())
api_hash = input("Enter API_HASH: ").strip()
mode = input("Select Type:\n[1] User Account (Phone + OTP)\n[2] Bot Token MTProto\nChoice (1/2): ").strip() or "1"

async def run():
    if lib == "1":
        if mode == "2":
            bot_token = input("Enter BOT_TOKEN: ").strip()
            client = Client("colab_bot", api_id=api_id, api_hash=api_hash, bot_token=bot_token, in_memory=True)
            await client.start()
            session = await client.export_session_string()
            me = await client.get_me()
            print(f"\n\033[1;32m[✓] Generated Bot Session for @{me.username}!\033[0m")
            print(f"\nSession String:\n\033[1;37m{session}\033[0m")
            await client.stop()
        else:
            phone = input("Enter Phone (+1234567890): ").strip()
            client = Client("colab_user", api_id=api_id, api_hash=api_hash, in_memory=True)
            await client.connect()
            sent = await client.send_code(phone)
            otp = input("Enter OTP sent to Telegram (e.g. 1 2 3 4 5): ").strip().replace(" ", "")
            try:
                await client.sign_in(phone, sent.phone_code_hash, otp)
            except Exception as e:
                if "2FA" in str(e) or "password" in str(e).lower() or "SessionPasswordNeeded" in str(e):
                    pwd = getpass.getpass("Enter 2FA Password: ").strip()
                    await client.check_password(pwd)
                else:
                    raise e
            session = await client.export_session_string()
            me = await client.get_me()
            print(f"\n\033[1;32m[✓] Pyrogram v2 Session Generated for {me.first_name}!\033[0m")
            print(f"\nSession String:\n\033[1;37m{session}\033[0m")
            await client.disconnect()
    else:
        client = TelegramClient(StringSession(), api_id, api_hash)
        if mode == "2":
            bot_token = input("Enter BOT_TOKEN: ").strip()
            await client.start(bot_token=bot_token)
            session = client.session.save()
            me = await client.get_me()
            print(f"\n\033[1;32m[✓] Generated Telethon Bot Session for @{me.username}!\033[0m")
            print(f"\nSession String:\n\033[1;37m{session}\033[0m")
            await client.disconnect()
        else:
            phone = input("Enter Phone (+1234567890): ").strip()
            await client.connect()
            sent = await client.send_code_request(phone)
            otp = input("Enter OTP sent to Telegram: ").strip().replace(" ", "")
            try:
                await client.sign_in(phone, otp, phone_code_hash=sent.phone_code_hash)
            except Exception as e:
                pwd = getpass.getpass("Enter 2FA Password: ").strip()
                await client.sign_in(password=pwd)
            session = client.session.save()
            me = await client.get_me()
            print(f"\n\033[1;32m[✓] Telethon Session Generated for {me.first_name}!\033[0m")
            print(f"\nSession String:\n\033[1;37m{session}\033[0m")
            await client.disconnect()

asyncio.get_event_loop().run_until_complete(run())
